# T11b — LSTM (Long Short-Term Memory)

Uses `deep_learning.py` for all shared setup, training and evaluation.

**Model:** LSTM — adds a cell state to handle long-range dependencies. Standard go-to for CMAPSS in the literature.

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

In [ ]:
from src.models.deep_learning import *
import torch.nn as nn

print(f"Device: {DEVICE}")


## 1. Load data & build windows

In [ ]:
train_df, test_df = load_data()
FEAT_COLS  = select_features(train_df)
N_FEATURES = len(FEAT_COLS)

X_train, y_train, X_val, y_val = engine_split(train_df, FEAT_COLS)
X_test, y_test = build_windows(test_df, FEAT_COLS, is_test=True)

train_loader, val_loader, test_loader = make_loaders(
    X_train, y_train, X_val, y_val, X_test, y_test
)


## 2. Model definition

LSTM — forget gate + input gate + cell state allow it to learn what to remember across long sequences.

In [ ]:
class LSTMModel(nn.Module):
    """
    LSTM for RUL regression.

    Architecture
    ------------
    LSTM (2 layers, hidden=64, batch_first=True)
    → take last hidden state (h_n)
    → Linear → scalar RUL prediction
    """
    def __init__(self, n_features, hidden_size=64, n_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = n_features,
            hidden_size = hidden_size,
            num_layers  = n_layers,
            batch_first = True,
            dropout     = dropout if n_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        last   = out[:, -1, :]
        return self.fc(last).squeeze(-1)

model = LSTMModel(N_FEATURES)
n_params = sum(p.numel() for p in model.parameters())
print(f"LSTM parameters: {n_params:,}")


## 3. Train

In [ ]:
model, train_losses, val_losses = train_model(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    epochs       = EPOCHS,
    model_name   = "LSTM",
)


## 4. Evaluate

In [ ]:
from src.evaluation.metrics import evaluate

y_true, y_pred = predict_test(model, test_loader)
results = evaluate(y_true, y_pred, model_name="LSTM")
print(results)


## 5. Plots

In [ ]:
plot_loss_curves(train_losses, val_losses, model_name="LSTM")
plot_predictions(y_true, y_pred, model_name="LSTM")
